In [1]:
import os
from openai import OpenAI
import rich

In [2]:
API_KEY = os.environ.get('OPENAI_API_KEY')
BASE_URL = os.environ.get('OPENAI_BASE_URL')
MODEL = "gpt-5.4"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Sending Tools information into API calls**

# Chat Completion API

https://platform.openai.com/docs/guides/function-calling?api-mode=chat

Defining the structure schema of the function to be passed as a tool in the API.

In [3]:
def get_weather_function_chat():
    return {
        "type": "function",
        "function": { # This property is removed from responses API
            "name": "get_weather",
            "description": "Get the weather for a location. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and Countery e.g Karachi, Pakistan"
                    }
                },
                "required": ["location"],
                "additionalProperties": False
            },
            "strict": True
        }
    }

In [7]:
response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "developer", "content": "你是玲娜贝儿，是我是私人天气顾问。"},
        {"role": "user", "content": "今天长沙的天气怎么样?"}
        # {"role": "user", "content": "NYC"}
    ],
    tools = [get_weather_function_chat()]
)

print(response.choices[0].message.content)
rich.print(response)
rich.print(response.choices[0].message.tool_calls)
print(response.choices[0].message.tool_calls[0].function.name)
print(response.choices[0].message.tool_calls[0].function.arguments)
print()
# Chat API provide value 'tool_calls' in finish_reason whenever a tool call is required
print("Finish Reason = ", response.choices[0].finish_reason)

None


ChatCompletion(
    id='resp_01cf2dc6753add7a0169d901f8c2c08194bab618bc83b62c89',
    choices=[
        Choice(
            finish_reason='tool_calls',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                refusal=None,
                role='assistant',
                annotations=None,
                audio=None,
                function_call=None,
                tool_calls=[
                    ChatCompletionMessageFunctionToolCall(
                        id='call_RcCaYaKfRj3uWCSokTwG1UX2',
                        function=Function(arguments='{"location":"长沙, 中国"}', name='get_weather'),
                        type='function'
                    )
                ],
                reasoning_content=None
            ),
            native_finish_reason='tool_calls'
        )
    ],
    created=1775829496,
    model='gpt-5.4',
    object='chat.completion',
    service_tier=None,
    system_fingerprint=None,
    usage=CompletionUsage(
        completion_tokens=44,
        prompt_tokens=102,
        total_tokens=146,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=None,
            audio_tokens=None,
            reasoning_tokens=22,
            rejected_prediction_tokens=None
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)
    )
)

[
    ChatCompletionMessageFunctionToolCall(
        id='call_RcCaYaKfRj3uWCSokTwG1UX2',
        function=Function(arguments='{"location":"长沙, 中国"}', name='get_weather'),
        type='function'
    )
]

get_weather
{"location":"长沙, 中国"}

Finish Reason =  tool_calls


# Responses API

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

Schema strucute is different in Responses API

In [5]:
def get_weather_function_response():
    return {
        "type": "function", # There is no function property in the response API
        "name": "get_weather",
        "description": "Get the weather for a location. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and Countery e.g Karachi, Pakistan"
                }
            },
            "required": ["location"],
            "additionalProperties": False
        },
        "strict": True
    }

In [8]:
response = openai.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": "今天长沙的天气怎么样?"}],
    tools = [get_weather_function_response()]
)

print(response.output_text)
rich.print(response)
rich.print("Status = ",response.status) # No indication of the tool call
rich.print(response.output[0]) # if you mention two cities in the input, you will get two outputs 0 and 1
print()
print("Name = ", response.output[0].name)
print("Type = ", response.output[0].type)
print("Arguments = ",response.output[0].arguments)

Response(
    id='resp_02ada732f5aacbaa0169d90242c4408195b8a22bf5c50e78f0',
    created_at=1775829570.0,
    error=None,
    incomplete_details=None,
    instructions=None,
    metadata={},
    model='gpt-5.4',
    object='response',
    output=[
        ResponseFunctionToolCall(
            arguments='{"location":"长沙, 中国"}',
            call_id='call_FIA1KLDc2fb1gXso9Zj8igGg',
            name='get_weather',
            type='function_call',
            id='fc_02ada732f5aacbaa0169d9024610888195b26e32019d995fad',
            status='completed'
        )
    ],
    parallel_tool_calls=True,
    temperature=1.0,
    tool_choice='auto',
    tools=[
        FunctionTool(
            name='get_weather',
            parameters={
                'type': 'object',
                'properties': {
                    'location': {'type': 'string', 'description': 'City and Countery e.g Karachi, Pakistan'}
                },
                'required': ['location'],
                'additionalProperties': False
            },
            strict=True,
            type='function',
            description="Get the weather for a location. Call this whenever you need to know the weather, for 
example when a customer asks 'What's the weather like in this city'"
        )
    ],
    top_p=0.98,
    background=False,
    conversation=None,
    max_output_tokens=None,
    max_tool_calls=None,
    previous_response_id=None,
    prompt=None,
    prompt_cache_key='fddcc08a-8555-4b20-82a2-59b51fcf933c',
    reasoning=Reasoning(effort='none', generate_summary=None, summary=None),
    safety_identifier='user-ESYHfImkGWTkSSTtTrBDgPiG',
    service_tier='default',
    status='completed',
    text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'),
    top_logprobs=0,
    truncation='disabled',
    usage=ResponseUsage(
        input_tokens=85,
        input_tokens_details=InputTokensDetails(cached_tokens=0),
        output_tokens=20,
        output_tokens_details=OutputTokensDetails(reasoning_tokens=0),
        total_tokens=105
    ),
    user=None,
    completed_at=1775829574,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    prompt_cache_retention=None,
    store=False,
    tool_usage={
        'image_gen': {
            'input_tokens': 0,
            'input_tokens_details': {'image_tokens': 0, 'text_tokens': 0},
            'output_tokens': 0,
            'output_tokens_details': {'image_tokens': 0, 'text_tokens': 0},
            'total_tokens': 0
        },
        'web_search': {'num_requests': 0}
    }
)

Status =  completed

ResponseFunctionToolCall(
    arguments='{"location":"长沙, 中国"}',
    call_id='call_FIA1KLDc2fb1gXso9Zj8igGg',
    name='get_weather',
    type='function_call',
    id='fc_02ada732f5aacbaa0169d9024610888195b26e32019d995fad',
    status='completed'
)


Name =  get_weather
Type =  function_call
Arguments =  {"location":"长沙, 中国"}
